# Kelvin transformation: the exterior source, the twisted pullback, and A-Phi at p = 2

Three measured results, locked by goldens in this repository and demonstrated
live in this notebook (every number below is recomputed when the notebook runs):

| # | question | answer | golden |
|---|---|---|---|
| 1 | What goes in the **Kelvin exterior** for a reduced potential when the background does not vanish at the Kelvin radius? | The 1-form Convention B rule is **exact**; dropping the source loses exactly **2/3**; differentiating the 0-form Convention B rule overshoots by exactly **4/3** | `validation_test/kelvin_source/test_kelvin_exterior_source_routes.py` |
| 2 | Does the **potential route** (`Omega_s`, T-Omega) work at all? | Yes -- via the genuine **twisted 0-form pullback** `Omega_s'(r') = -Omega_s(k(r'))`, which is gradient-consistent with the twisted 1-form rule to machine precision. Decaying sources are regular at the Kelvin centre; a uniform background has no bounded 0-form image (physics, not a defect) | `tests/test_reduced_potential_background.py` |
| 3 | Does the **A-Phi (A-V) eddy formulation + Periodic Kelvin work at p = 2**? | Yes -- 0.05% vs the analytic sphere. The plain A-method is **p-saturated** (~0.45%) because `nograds=True` removes the charge-conservation test functions; the explicit Phi block restores them | `validation_test/kelvin_source/test_aphi_kelvin_eddy.py` |

The unifying language is **premetric electromagnetism**: form *degree* and
orientation *parity* are independent axes, the Kelvin inversion is
orientation-reversing (`det Dk = -R^6/rho'^6 < 0`), and **twisted** quantities
pick up the extra sign `s_k = sgn(det Dk) = -1` that **straight** quantities
do not:

```
twisted  (extra -1):  phi_m (0-form), h (1-form), d, j (2-forms), rho, U_m (3-forms)
straight (no sign) :  V (0-form), a, e (1-forms), b (2-form)
```

References: `docs/kelvin/KELVIN_TRANSFORMATION.md` sections 2.3 / 7.4 / 7.6 / 7.9,
the lab note <https://www.ele.kindai.ac.jp/laboratory/sugahara/elemag/geometry09.php>
(sign table), and the MCP topics `kelvin_transformation("exterior_source")`,
`kelvin_transformation("verified_recipe")`,
`differential_forms_maxwell("twisted")`.


In [1]:
from pathlib import Path
import datetime as dt
import importlib.util
import json
import platform
import sys

# Locate the repo root from the notebook directory (docs/kelvin/)
NOTEBOOK_DIR = Path.cwd()
REPO = next(p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.resolve().parents]
            if (p / "src" / "radia").exists())
sys.path.insert(0, str(REPO / "src"))


def load_module(rel_path, name):
    spec = importlib.util.spec_from_file_location(name, REPO / rel_path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


import ngsolve
import radia

versions = {
    "radia_version": getattr(radia, "__version__", "unknown"),
    "ngsolve_version": ngsolve.__version__,
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
}
print(json.dumps(versions, indent=1))

{
 "radia_version": "4.95.23",
 "ngsolve_version": "6.2.2604",
 "python_version": "3.12.10",
 "platform": "Windows-2022Server-10.0.20348-SP0"
}


## Part 1 -- the background field inside the Kelvin exterior

A magnetic sphere ($\mu_r = 100$, $a = 0.5$ m) sits in a uniform applied field
$H_0 \hat z$, solved with the two-sphere Periodic Kelvin open boundary and a
reduced scalar potential $H = H_s - \nabla\Omega$. The analytic interior field
is $H_\mathrm{in} = 3H_0/(\mu_r+2)$. **One mesh, one bilinear form** -- the
three routes differ *only* in what $H_s$ is inside the Kelvin exterior:

| route | $H_s'$ in the Kelvin exterior |
|---|---|
| **Z** | $0$ (source dropped -- what a solver does when it excludes the Kelvin region from the source projection) |
| **B1** | $-(\rho'/R)^2 H_0\hat z$ -- the 1-form **Convention B** rule (`make_reduced_potential_background_cf`) |
| **B0** | $-\nabla\Omega_s'$ with $\Omega_s' = -(\rho'/R)^2\,\Omega_s(\text{local})$ -- the (invalid) "0-form Convention B" |

Why the minus sign in Convention B at all: $H$ is a **twisted 1-form** and the
Kelvin map reverses orientation. Energy is orientation-blind (two sign flips
cancel), which is why the material rule $\nu' = (\rho'/R)^2\nu_0$ carries no sign.

In [2]:
routes = load_module(
    "validation_test/kelvin_source/test_kelvin_exterior_source_routes.py",
    "kelvin_routes")

res1 = routes._results()
m_ana = routes.H_ANALYTIC
rows = []
for key, label in (("Z", "H_s' = 0 (source dropped)"),
                   ("B1", "1-form Convention B"),
                   ("B0", "-grad(0-form Convention B)")):
    hz = res1[key]
    err = 100.0 * (hz / m_ana - 1.0)
    rows.append((key, label, hz, err))
    print(f"  {key:3s} {label:32s} H_z = {hz:+.6f}   err = {err:+8.3f}%")
ratio_z = res1["Z"] / res1["B1"]
ratio_b0 = res1["B0"] / res1["B1"]
print(f"\n  analytic H_in = {m_ana:.6f}")
print(f"  Z  / B1 = {ratio_z:.6f}   (2/3 = {2/3:.6f})")
print(f"  B0 / B1 = {ratio_b0:.6f}   (4/3 = {4/3:.6f})")
assert abs(ratio_z - 2/3) < 1e-3 and abs(ratio_b0 - 4/3) < 1e-3
part1 = {"H_analytic": m_ana,
         "routes": {k: {"H_z": v, "err_pct": 100*(v/m_ana-1)}
                    for k, v in ((r[0], r[2]) for r in rows)},
         "ratio_Z_over_B1": ratio_z, "ratio_B0_over_B1": ratio_b0}

  Z   H_s' = 0 (source dropped)        H_z = +0.019756   err =  -32.830%
  B1  1-form Convention B              H_z = +0.029630   err =   +0.743%
  B0  -grad(0-form Convention B)       H_z = +0.039508   err =  +34.329%

  analytic H_in = 0.029412
  Z  / B1 = 0.666748   (2/3 = 0.666667)
  B0 / B1 = 1.333384   (4/3 = 1.333333)


The factors **2/3** and **4/3** are mesh-independent (at $p=3$: 0.666669 /
1.333338) -- they are *structural*, not discretisation error. Route B1
converges to the analytic answer (**+0.002%** at $p=3$, recorded in the
golden). Readings:

1. the 1-form Convention B rule is the correct way to carry a non-decaying
   background into the Kelvin exterior;
2. dropping the exterior source is **not** a small approximation -- it loses
   exactly one third of the field;
3. "0-form Convention B" is not a pullback of anything: a 0-form pullback
   carries **no** metric factor. Differentiating it overshoots by 4/3 --
   which is Part 2's subject.

## Part 2 -- the twisted 0-form pullback: the rule that makes $\Omega_s$ work

The magnetic scalar potential is a **twisted 0-form**, so its genuine pullback
under the (orientation-reversing) Kelvin map carries the twist sign and **no
metric factor**:

$$\Omega_s'(r') = -\,\Omega_s\!\big(k(r')\big), \qquad
  H_s'(r') = -\Big(\tfrac{R}{\rho'}\Big)^2 (I - 2nn^\top)\, H_s\!\big(k(r')\big)$$

(radial component keeps its sign, tangential components flip). Because
pullback commutes with the exterior derivative and both quantities carry the
*same* twist factor, $H_s' = -\nabla'\Omega_s'$ **exactly** -- the property
Convention B lacks (`curl` of the Convention-B field is
$(2H_0/R^2)(-y',x',0) \ne 0$, so it has no potential at all).

APIs: `radia.kelvin_material.make_kelvin_aware_Omega_s_cf` /
`make_kelvin_aware_H_s_cf`. The four cells of evidence (each executes the
corresponding contract lock):

In [3]:
twisted = load_module("tests/test_reduced_potential_background.py",
                      "reduced_potential_tests")
twisted.test_twisted_0form_pullback_formula()
twisted.test_twisted_pullback_family_is_gradient_consistent()

=== Test 9: twisted 0-form pullback formula ===
  r'=(+0.40,+0.20,+0.30): got=-1.04926150e-02, want=-1.04926150e-02, rel=1.65e-16
  r'=(+0.30,-0.25,+0.20): got=-1.02770764e-02, want=-1.02770764e-02, rel=3.38e-16
  r'=(-0.30,+0.15,-0.30): got=+7.73100594e-03, want=+7.73100594e-03, rel=2.24e-16
  [OK] Omega_s' = -Omega_s(k(r'))

=== Test 10: twisted pullback is gradient-consistent ===
  (3.4, 0.2, 0.3): -grad(Om') = ('+2.87892e-02', '-1.24763e-02', '+4.07878e-02')
        H_s'       = ('+2.87892e-02', '-1.24763e-02', '+4.07878e-02')  rel=8.88e-11
  (3.3, -0.25, 0.2): -grad(Om') = ('+3.09164e-02', '-2.23252e-02', '+3.93000e-02')
        H_s'       = ('+3.09164e-02', '-2.23252e-02', '+3.93000e-02')  rel=1.38e-10
  (2.7, 0.15, -0.3): -grad(Om') = ('+1.84451e-02', '-1.00344e-02', '+2.26169e-02')
        H_s'       = ('+1.84451e-02', '-1.00344e-02', '+2.26169e-02')  rel=3.04e-11
  max rel = 1.38e-10
  [OK] -grad(Omega_s') == H_s' for the twisted pullback family



In [4]:
twisted.test_twisted_0form_regular_at_offset_for_decaying_source()
twisted.test_twisted_0form_diverges_for_a_uniform_background()
part2 = {"gradient_consistency": "H_s' == -grad(Omega_s') to ~1e-10 (FD-limited)",
         "decaying_source": "regular at rho'->0, Omega_s' = O(rho'^2)",
         "uniform_background": "diverges like R^2/rho'^2 (no bounded 0-form image)"}

=== Test 11: twisted 0-form regular at the offset (decaying) ===


  rho'= 0.40: |Omega_s'| = 7.031474e-03
  rho'= 0.20: |Omega_s'| = 1.676622e-03
  rho'= 0.10: |Omega_s'| = 4.085902e-04
  rho'= 0.05: |Omega_s'| = 1.008126e-04
  rho'= 0.02: |Omega_s'| = 1.600139e-05
  [OK] decaying source -> regular at rho' = 0

=== Test 12: uniform background has no bounded 0-form image ===
  rho'= 0.40: |Omega_s'| = 2.500000e+00
  rho'= 0.20: |Omega_s'| = 5.000000e+00
  rho'= 0.10: |Omega_s'| = 1.000000e+01
  rho'= 0.05: |Omega_s'| = 2.000000e+01
  [OK] uniform background diverges -> use the 1-form route



**Which route your source forces** (the practical table):

| source at infinity | 0-form image at $\rho'\to 0$ | usable route |
|---|---|---|
| **decays** (real coil, dipole $1/r^3$) | regular, $\Omega_s' = O(\rho'^2)$ | **potential route works** -- T-Omega + Kelvin is fine |
| **uniform at infinity** | diverges like $R^2/\rho'^2$ | field route only (1-form Convention B, Part 1) |

The divergence is physics -- the uniform-field potential is unbounded at
infinity and $\rho'=0$ *is* infinity -- not a defect of the rule.

## Part 3 -- A-Phi (A-V) + Periodic Kelvin at p = 2

Conducting non-magnetic sphere ($a/\delta = 2$) in a uniform harmonic field,
analytic induced moment (Smythe)
$m_z = -2\pi a^3 (B_0/\mu_0)\,[\,1 - (3/x)\coth x + 3/x^2\,]$, $x=(1+j)a/\delta$.
Both lanes share one mesh and every `verified_recipe` element
(`nograds=True` $\to$ `Periodic`, gauge regularization on non-Kelvin
materials, `bonus_intorder=4`):

* **A\*** (plain A-method): $J = -s\sigma(A_s + A_r)$
* **A-Phi** (mixed HCurl $\times$ H1): $J = -s\sigma(A_s + A_r + \nabla W)$,
  $W = V/s$, with the one-product symmetric form
  $s\,\sigma\,(A_r+\nabla W)\cdot(\bar A'+\nabla \bar q)$

The source needs **no Kelvin-exterior background** here:
$\nabla\times(\nu_0\nabla\times A_s)=0$ for uniform $B$, so the source lives
entirely in the conductor mass term and the reaction field decays -- the
decaying case of Part 2.

In [5]:
aphi = load_module("validation_test/kelvin_source/test_aphi_kelvin_eddy.py",
                   "aphi_kelvin")
res3 = aphi._results()   # solves (p=1,2) x (A*, A-Phi) on one mesh
m_ana3 = res3["m_ana"]
print(f"m_analytic = {m_ana3:.6e}\n")
print(f"{'p':>2s} {'A* rel':>10s} {'A-Phi rel':>10s}   FES verify (slaved / ratio)")
part3_rows = {}
for p in (1, 2):
    slaved, ratio = res3[f"fes_p{p}"]
    ra = abs(res3[f"astar_p{p}"] - m_ana3) / abs(m_ana3)
    rf = abs(res3[f"aphi_p{p}"] - m_ana3) / abs(m_ana3)
    part3_rows[p] = {"astar_rel_pct": 100*ra, "aphi_rel_pct": 100*rf,
                     "slaved": slaved, "kelvin_ratio": ratio}
    print(f"{p:2d} {ra*100:9.3f}% {rf*100:9.3f}%   {slaved} / {ratio:.6f}")
factor = part3_rows[2]["astar_rel_pct"] / part3_rows[2]["aphi_rel_pct"]
print(f"\nA-Phi beats A* at p=2 by a factor {factor:.1f}")
part3 = {"m_analytic": [m_ana3.real, m_ana3.imag], "by_order": part3_rows,
         "p2_factor_aphi_over_astar": factor}

m_analytic = -7.927178e+04-1.077368e+05j

 p     A* rel  A-Phi rel   FES verify (slaved / ratio)
 1     2.889%     2.889%   1659 / 1.000000
 2     0.473%     0.053%   3871 / 1.000000

A-Phi beats A* at p=2 by a factor 8.9


Recorded extensions from the golden (measured 2026-07-25, not re-run here for
runtime): at $p=3$ the plain A-method **saturates** (0.442% vs 0.473% at
$p=2$) while A-Phi keeps converging (0.005%); under $h$-refinement at fixed
$p=2$ the A* error decays only $\sim O(h)$ (0.473 $\to$ 0.243 $\to$ 0.141%)
while A-Phi drops fast (0.053 $\to$ 0.011 $\to$ 0.004%, factor 9x $\to$ 22x
$\to$ 35x). The gauge-regularization coefficient ($10^{-5}..10^{-10}$) changes
neither lane to 4 digits.

**Why**: `nograds=True` -- which the Periodic-Kelvin recipe *requires* --
removes the gradient test functions that would enforce discrete charge
conservation $\nabla\cdot(\sigma A)=0$; the explicit Phi block restores them.
The continuous $V$ is exactly zero for this axisymmetric problem, so the
entire A\*/A-Phi gap is *discrete* charge-conservation error: with the plain
A-method only $h$ buys accuracy, with A-Phi $p$-refinement pays.

**Rule of thumb**: SIBC-driven problems (no volume eddy mass, e.g.
`calc_fem_kelvin.py`) stay on the plain A-method; a **volume conductor at
$p\ge 2$ takes the A-Phi block**.

In [6]:
# Persist the domain results JSON next to this notebook (Data Persistence policy)
results = {
    "schema": "radia.kelvin_exterior_source_and_aphi.v1",
    "generated_at_utc": dt.datetime.now(dt.timezone.utc)
        .isoformat(timespec="seconds").replace("+00:00", "Z"),
    **versions,
    "part1_exterior_source_routes": part1,
    "part2_twisted_pullback": part2,
    "part3_aphi_vs_astar": part3,
    "goldens": [
        "validation_test/kelvin_source/test_kelvin_exterior_source_routes.py",
        "tests/test_reduced_potential_background.py",
        "validation_test/kelvin_source/test_aphi_kelvin_eddy.py",
    ],
}
out = NOTEBOOK_DIR / "kelvin_exterior_source_and_aphi_results.json"
out.write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")
print("wrote", out.name)
print(json.dumps({k: results[k] for k in
                  ("schema", "generated_at_utc", "radia_version")}, indent=1))

wrote kelvin_exterior_source_and_aphi_results.json
{
 "schema": "radia.kelvin_exterior_source_and_aphi.v1",
 "generated_at_utc": "2026-07-25T10:25:56Z",
 "radia_version": "4.95.23"
}


## Conclusions

1. **Exterior source**: a non-decaying background *must* be carried into the
   Kelvin exterior by the 1-form Convention B rule -- dropping it costs
   exactly 2/3, mislabeling the 0-form costs exactly 4/3.
2. **Potential route**: the twisted 0-form pullback
   $\Omega_s' = -\Omega_s(k(r'))$ is gradient-consistent with the twisted
   1-form rule to machine precision, regular for decaying sources -- so
   T-Omega + Kelvin works for real coils; only a uniform background at
   infinity is forced onto the field route.
3. **A-Phi at p = 2**: yes -- 0.05% vs analytic, an order of magnitude better
   than the plain A-method, whose remaining error is discrete
   charge-conservation error that $p$-refinement cannot remove.

All three statements are enforced by fast pytest goldens; this notebook is the
rendered explanation, re-running the same locked code.